In [30]:
import warnings
warnings.filterwarnings('ignore')

In [31]:
import os
from dotenv import load_dotenv
_=load_dotenv()

In [32]:
profile = {
    "name": "Arshad",
    "full_name": "Mohammed Arshad",
    "user_profile_background": "Senior  AI engineer leading a team of 7 junior analysts",
}

In [33]:
prompt_instructions = {
    "triage_rules": {
        "ignore": "Marketing newsletters, spam emails, mass company announcements",
        "notify": "Team member out sick, build system notifications, project status updates",
        "respond": "Direct questions from team members, meeting requests, critical bug reports",
    },
    "agent_instructions": "Use these tools when appropriate to help manage Arshad's tasks efficiently."
}

In [34]:
email = {
    "from": "Ananya Gupta <ananya.gupta@company.com>",
    "to": "Mohammed Arshad <Arshad.mohd@company.co>",
    "subject": "Clarification on KPI definitions",
    "body": """
Hi Arshad,

While preparing the dashboard, I noticed some ambiguity in KPI definitions.

Could you clarify how we are calculating:
- Customer Retention Rate
- Monthly Active Users (MAU)

This will help ensure consistency in reporting.

Regards,
Ananya
"""
}

In [35]:
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model

In [36]:
llm = init_chat_model("groq:llama-3.1-8b-instant")

In [37]:
class Router(BaseModel):
    """Analyze the unread email and route it according to its content."""

    reasoning: str = Field(
        description="Step-by-step reasoning behind the classification."
    )
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="The classification of an email: 'ignore' for irrelevant emails, "
        "'notify' for important information that doesn't need a response, "
        "'respond' for emails that need a reply",
    )

In [38]:
llm_router = llm.with_structured_output(Router)

In [39]:
from prompts import triage_system_prompt, triage_user_prompt

In [40]:
system_prompt = triage_system_prompt.format(
    full_name=profile["full_name"],
    name=profile["name"],
    examples=None,
    user_profile_background=profile["user_profile_background"],
    triage_no=prompt_instructions["triage_rules"]["ignore"],
    triage_notify=prompt_instructions["triage_rules"]["notify"],
    triage_email=prompt_instructions["triage_rules"]["respond"],
)

In [41]:
user_prompt = triage_user_prompt.format(
    author=email["from"],
    to=email["to"],
    subject=email["subject"],
    email_thread=email["body"],
)

In [42]:
result = llm_router.invoke(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
)

In [43]:
print(result)

reasoning='The email is a direct question from a team member, so it requires a response from Arshad to provide clarity on the KPI definitions.' classification='respond'


###

### Main agent and tools

In [44]:
from langchain_core.tools import tool

In [45]:
from langchain.tools import tool

@tool
def check_calendar_availability(day: str) -> str:
    """ALWAYS use this tool when the user asks about availability, schedule, or free time for a day."""
    
    availability = {
        "monday": ["10 AM", "3 PM"],
        "tuesday": ["9 AM", "2 PM", "4 PM"],
        "wednesday": ["11 AM", "1 PM"],
    }
    
    slots = availability.get(day.lower(), [])
    
    if not slots:
        return f"No available slots on {day}"
    
    return f"Available slots on {day}: {', '.join(slots)}"


@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Send an email."""
    return f"Email sent to {to} with subject '{subject}'"


@tool
def schedule_meeting(attendees: list, subject: str, duration_minutes: int, preferred_day: str) -> str:
    """Schedule a meeting."""
    return f"Meeting scheduled with {attendees} on {preferred_day} for {duration_minutes} minutes"

@tool
def notify(message: str) -> str:
    """Use this tool to notify the user about important information that does not require a response."""
    return f"Notification: {message}"

###

### Prompt

In [46]:
from prompts import agent_system_prompt
def create_prompt(state):
    return [
        {
            "role": "system", 
            "content": agent_system_prompt.format(
                instructions=prompt_instructions["agent_instructions"],
                **profile
                )
        }
    ] + state['messages']

In [47]:
print(agent_system_prompt)


< Role >
You are {full_name}'s executive assistant.
</ Role >

< Tools >
You have access to tools:
- write_email
- schedule_meeting
- check_calendar_availability
</ Tools >

< Instructions >
{instructions}

STRICT RULES:
- If a tool is relevant, you MUST call it immediately.
- NEVER say "let me check" or "let me think".
- NEVER delay action.
- NEVER loop.

- After calling a tool, immediately return the final answer.
- DO NOT generate intermediate reasoning.

- Your output should always be either:
  1. A tool call
  2. A final answer

Be direct and efficient.
</ Instructions >



In [48]:
from langchain.agents import create_agent

In [49]:
tools=[write_email, schedule_meeting, check_calendar_availability]

In [53]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model="groq:llama-3.1-8b-instant",
    tools=tools,
    prompt=create_prompt,  
)

In [54]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is my availability for tuesday?"}]}
)

In [55]:
response["messages"][-1].pretty_print()

================================== Ai Message ==================================

Available slots on Tuesday: 9 AM, 2 PM, 4 PM
